In [ ]:
# Databricks notebook source
"""Load data from the Landing volume into a Bronze Delta table.

Expected notebook parameters (widgets):
  - source_system: e.g. adls_csv, neon_postgres
  - table_name: source/table name, e.g. lab_results
  - landing_timestamp: exact Landing folder timestamp, e.g. 2026-10-23_01:18:21
  - source_file_format: csv or parquet (optional; inferred from files if blank)
"""

from pyspark.sql.functions import current_timestamp, lit

# -----------------------------------------------------------------------------
# 1. Read pipeline parameters
# -----------------------------------------------------------------------------
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

source_system = dbutils.widgets.get("source_system").strip()
table_name = dbutils.widgets.get("table_name").strip()
landing_timestamp = dbutils.widgets.get("landing_timestamp").strip()
source_file_format = dbutils.widgets.get("source_file_format").strip().lower()

# These parameters are required to build the exact Landing folder.
missing_parameters = [
    name
    for name, value in {
        "source_system": source_system,
        "table_name": table_name,
        "landing_timestamp": landing_timestamp,
    }.items()
    if not value
]
if missing_parameters:
    raise ValueError(
        "Missing required notebook parameter(s): "
        + ", ".join(missing_parameters)
        + ". Check the parameter mapping in the pipeline."
    )

landing_path = (
    f"/Volumes/adb_healthy/landing/landing_volumne/"
    f"{source_system}/{table_name}/{landing_timestamp}"
)

print("Notebook parameters:")
print(f"  source_system      = [{source_system}]")
print(f"  table_name         = [{table_name}]")
print(f"  landing_timestamp  = [{landing_timestamp}]")
print(f"  source_file_format = [{source_file_format}]")
print(f"  landing_path       = [{landing_path}]")

In [ ]:
# -----------------------------------------------------------------------------
# 2. Verify the exact Landing folder exists
# -----------------------------------------------------------------------------
try:
    landing_entries = dbutils.fs.ls(landing_path)
except Exception as exc:
    # List parent folder when possible to make timestamp mismatches easier to debug.
    parent_path = (
        f"/Volumes/adb_healthy/landing/landing_volumne/"
        f"{source_system}/{table_name}/"
    )
    try:
        available_folders = [entry.name for entry in dbutils.fs.ls(parent_path)]
    except Exception:
        available_folders = []

    raise FileNotFoundError(
        f"Landing folder does not exist: {landing_path}. "
        f"Available folders under {parent_path}: {available_folders}. "
        "Pass the exact landing_timestamp created by the Landing ingestion step."
    ) from exc

if not landing_entries:
    raise FileNotFoundError(f"Landing folder is empty: {landing_path}")

print("Landing folder exists. Entries:")
for entry in landing_entries:
    print(f"  {entry.name} (size={entry.size})")

In [ ]:
# -----------------------------------------------------------------------------
# 3. Resolve source file format
# -----------------------------------------------------------------------------
# Prefer metadata supplied by the pipeline. If it is blank, infer it from the
# files in this exact folder so a missing optional parameter does not cause an
# opaque 'Unsupported source_file_format' error.
if not source_file_format:
    file_names = [entry.name.lower() for entry in landing_entries if entry.size > 0]
    has_csv = any(name.endswith(".csv") for name in file_names)
    has_parquet = any(name.endswith(".parquet") for name in file_names)

    if has_csv and not has_parquet:
        source_file_format = "csv"
    elif has_parquet and not has_csv:
        source_file_format = "parquet"
    elif has_csv and has_parquet:
        raise ValueError(
            f"Both CSV and Parquet files were found in {landing_path}. "
            "Set source_file_format explicitly in the pipeline."
        )
    else:
        raise ValueError(
            f"source_file_format was not provided and could not be inferred from "
            f"the files in {landing_path}. Set it to 'csv' or 'parquet' in the pipeline."
        )

if source_file_format not in {"csv", "parquet"}:
    raise ValueError(
        f"Unsupported source_file_format: [{source_file_format}]. "
        "Supported values are 'csv' and 'parquet'."
    )

print(f"Resolved source_file_format = [{source_file_format}]")

In [ ]:
# -----------------------------------------------------------------------------
# 4. Read Landing data
# -----------------------------------------------------------------------------
if source_file_format == "csv":
    landing_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(landing_path)
    )
elif source_file_format == "parquet":
    landing_df = spark.read.parquet(landing_path)

print(f"Loaded Landing data for table: {table_name}")
display(landing_df)


In [ ]:
# -----------------------------------------------------------------------------
# 5. Add audit columns
# -----------------------------------------------------------------------------
updated_df = (
    landing_df
    .withColumn("landing_timestamp", lit(landing_timestamp))
    .withColumn("insert_timestamp", current_timestamp())
)

In [ ]:
# -----------------------------------------------------------------------------
# 6. Append to Bronze table
# -----------------------------------------------------------------------------
bronze_table = f"adb_healthy.bronze.{table_name}"
print(f"Appending data to Bronze table: {bronze_table}")
updated_df.write.mode("append").saveAsTable(bronze_table)

print(f"Successfully appended Landing data into {bronze_table}")